In [1]:
import sys
sys.path.append('/opt/homebrew/Cellar/apache-spark/3.5.5/libexec/python')
sys.path.append('/opt/homebrew/Cellar/apache-spark/3.5.5/libexec/python/lib/py4j-0.10.9.7-src.zip')

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/09 22:51:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Question 1: Install Spark and PySpark

Install Spark

Run PySpark

Create a local spark session

Execute spark.version.spark.version

What's the output?

In [4]:
spark.version

'3.5.5'

### Question 2: Yellow October 2024

Read the October 2024 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [5]:
# https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [6]:
filename = 'yellow_tripdata_2024-10.parquet'

In [7]:
df_yellow = spark.read.parquet(filename)

In [8]:
df_yellow = df_yellow.repartition(4)

In [9]:
df_yellow.write.parquet('yellow_2024-10', mode="overwrite")

In [10]:
# A: 22.4 MB

### Question 3: Count records

How many taxi trips were there on the 15th of October?

Consider only trips that started on the 15th of October.

In [11]:
df_yellow.createOrReplaceTempView('yellow')

In [12]:
spark.sql(
    """
    SELECT COUNT(*) FROM yellow WHERE DATE(tpep_pickup_datetime) = '2024-10-15'
    """
).show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



In [13]:
# A: 128893

### Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

In [14]:
df_yellow \
    .withColumn('duration', 
                (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600) \
    .withColumn('pickup_date', F.to_date('tpep_pickup_datetime')) \
    .groupBy('pickup_date') \
    .agg(F.max('duration').alias('max_duration')) \
    .orderBy(F.desc('max_duration')) \
    .limit(5) \
    .show()

[Stage 10:====================================>                     (5 + 3) / 8]

+-----------+------------------+
|pickup_date|      max_duration|
+-----------+------------------+
| 2024-10-16|162.61777777777777|
| 2024-10-03|           143.325|
| 2024-10-22|137.76055555555556|
| 2024-10-18|114.83472222222223|
| 2024-10-21| 89.89833333333333|
+-----------+------------------+



In [15]:
# A: 162

### Question 5: User Interface

Spark’s User Interface which shows the application's dashboard runs on which local port?

http://localhost:4040/jobs/

In [17]:
# A: 4040

### Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

In [20]:
df_zones = spark.read.parquet('zones')
df_zones.createOrReplaceTempView('zones')

In [23]:
spark.sql(
    """
    SELECT 
      zones.Zone,
      COUNT(*) AS pickup_count
    FROM 
      yellow
    LEFT JOIN
      zones
    ON 
      yellow.PULocationID = zones.LocationID
    GROUP BY
      zones.Zone
    ORDER BY
      pickup_count
    LIMIT
      5
    """
).show()

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
|       Rikers Island|           2|
|       Arden Heights|           2|
|         Jamaica Bay|           3|
| Green-Wood Cemetery|           3|
+--------------------+------------+



In [24]:
# A: Governor's Island/Ellis Island/Liberty Island